In [1]:
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import mysql.connector
from mysql.connector import Error

## A) Con el archivo titanic_data.csv realiza lo siguiente: 
### 1. Carga el contenido del archivo.

In [2]:
df = pd.read_csv('titanic_data.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


### 2. Añade dos columnas nuevas al dataframe (copias de age) y llena los datos vacíos utilizando promedio y moda. Es decir, el dataframe tendrá una columna nueva con los vacíos llenos por promedio y otra columna con los vacíos llenos por moda. 

In [3]:
promedio = df['Age'].mean()
moda = df['Age'].mode()[0]

df['Age_promedio'] = df['Age'].fillna(promedio)
df['Age_moda'] = df['Age'].fillna(moda)

### 3. Realiza la limpieza de los datos. 

In [4]:
# Eliminar duplicados
df = df.drop_duplicates()
# Eliminar nulos
df = df.dropna(how='all')

### 4. Valida los datos del dataframe considerando que solo puede haber dos valores en la columna Sex, que la edad debe estar entre 0 y 100 (usa la columna llenada con promedio) y que solo existen tres tipos de pasajeros (1, 2 y 3). Cualquier registro que no cumpla esto deberá ser eliminado. 

In [5]:
# Validación de sexo
condicion_sexo = (df['Sex'].isin(['male', 'female']))
# Validación de edad
condicion_edad = (df['Age_promedio'] >= 0) & (df['Age_promedio'] <= 100)
# Validacion de tipos de pasajeros
condicion_pasajeros = (df['Pclass'].isin([1, 2, 3]))
# Filtrado
df = df[condicion_sexo & condicion_edad & condicion_pasajeros]

### 5. Crea un modelo de base de datos que permita cargar los datos del pasajero. Este modelo debe incluir un identificador, el nombre, el sexo, la edad, el tipo de pasejero y si sobrevivió o no. 
### 6. Prepara los datos del dataframe para cargar el modelo que creaste previamente. 
### 7. Realiza la carga de datos en la base. 

In [6]:
config_db = {
    'host': 'localhost', 
    'user': 'root',
    'password': 'root',
    'port': 3306,
    'database': 'ordinario_u2',
    'charset': 'utf8mb4', 
    'use_unicode': True
}

# Realizar la conexión
try:
    connection = mysql.connector.connect(**config_db)
    if connection.is_connected():
        print('Conexion exitosa')
except Error as e:
    print(f'Error al conectar: {e}')

Conexion exitosa


In [7]:
# Insertar los datos
cursor = connection.cursor()
for i, registro in df.iterrows():
    nombre = registro['Name']
    sexo = registro['Sex']
    edad = registro['Age_promedio']
    tipo_pasajero = registro['Pclass']
    sobrevivio = registro['Survived']
    cursor.execute('INSERT IGNORE INTO pasajeros_titanic(nombre, sexo, edad, tipo_pasajero, sobrevivio) VALUES (%s, %s, %s, %s, %s)', 
                   (nombre, sexo, edad, tipo_pasajero, sobrevivio))
    connection.commit()

## B) Utilizando el archivo salarysurvey.csv realiza lo siguiente: 

In [8]:
df_survey = pd.read_csv('salarysurvey.csv')
df_survey

,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,4/27/2021 11:02:41,25-34,Nonprofits,Program Manager,NaN,"62,000",3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,4/27/2021 11:02:42,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,"60,000",7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27826,6/9/2022 9:45:49,18-24,Property or Construction,Environmental Technician,Petroleum Restoration,43000,5500.0,USD,NaN,NaN,United States,Florida,Tampa,2 - 4 years,2 - 4 years,College degree,Man,White
27827,6/10/2022 14:19:14,25-34,Business or Consulting,Analyst,NaN,75000,0.0,USD,NaN,NaN,United States of America,Virginia,"Richmond, VA",8 - 10 years,5-7 years,Master's degree,Woman,White
27828,6/12/2022 9:00:54,45-54,Computing or Tech,Technical Account MANAGER,NaN,75000,NaN,GBP,NaN,NaN,UK,NaN,London,11 - 20 years,8 - 10 years,College degree,Other or prefer not to answer,Another option not listed here or prefer not t...
27829,6/13/2022 4:47:22,25-34,Homemaker,Homemaker,NaN,0,0.0,USD,NaN,NaN,United States,California,Morgan Hill,5-7 years,2 - 4 years,College degree,Woman,White


### 1. ¿Cuáles son los distintos de Job title existen en el archivo? 

In [9]:
df_tipos = df_survey.groupby('Job title').size().reset_index(name='Cantidad')
df_tipos

,Job title,Cantidad
0,Analyst,1
1,Brand Manager,1
2,Business Systems Analyst,1
3,CAP team,1
4,Cloud Architect,1
...,...,...
14235,web developer,1
14236,workers comp case manager,1
14237,writer,2
14238,writer/editor,1


### 2. ¿Cuántos registros no se encuentran en los Estados Unidos?

In [10]:
(df_survey['What country do you work in?'] != 'United States').sum()

np.int64(18904)

### 3. ¿Cuántos registros son de personas no binarias con grado de maestría? 

In [11]:
( (df_survey['What is your highest level of education completed?'] == "Master's degree") & (df_survey['What is your gender?'] == 'Non-binary')  ).sum()

np.int64(213)

### 4. Muestra los registros 100 a 200 solo con las columnas de Job title y How old are you?

In [12]:
df_survey.iloc[100:201, [1, 3]]

,How old are you?,Job title
100,35-44,Production Co-ordinator
101,25-34,Administrative Support Assistant
102,25-34,Administrative Officer
103,45-54,Executive Secretary/Secretary to Board of Scho...
104,35-44,Senior Project Manager
...,...,...
196,25-34,Product Engineer
197,35-44,Senior Technical Writer
198,35-44,Senior Director
199,35-44,Hotel Sales Manager
